# **Project: Long Hair Identification**


**Problem Statement:**  In this task, you will develop a feature to detect a person with long hair as female, even if they are male, and detect a short-haired female as male, even if they are female. The model should work exclusively for individuals aged between 20 and 30. If an image of a person outside this age range (below 20 or above 30) is uploaded, the model should correctly predict their gender regardless of hair length. Guidelines: This task is designed to test your logic-building and problem-solving skills. We encourage you to embrace the challenge and view it as an opportunity to grow. Please create your own machine learning model and ensure it includes a graphical user interface (GUI). While accuracy is important, we will evaluate your work based on the overall performance of your model and the successful functionality of your GUI. Good luck and enjoy the process!

# Downloading Data

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ipythonx/celebamaskhq")

print("Path to dataset files:", path)

# Checking Folder Structure

In [ ]:
import os

def list_folder_structure(startpath, max_depth=2):
    for root, dirs, files in os.walk(startpath):
        # Calculate depth to avoid listing thousands of files
        depth = root.replace(startpath, '').count(os.sep)
        if depth > max_depth:
            continue

        indent = ' ' * 4 * depth
        print(f'{indent}{os.path.basename(root)}/')

        # Show only first 3 items in each folder for brevity
        subindent = ' ' * 4 * (depth + 1)
        for f in files[:3]:
            print(f'{subindent}{f}')
        if len(files) > 3:
            print(f'{subindent}...(and {len(files)-3} more files)')

# Define the base path from your download output
dataset_path = '/root/.cache/kagglehub/datasets/ipythonx/celebamaskhq/versions/1'

print("Structure of CelebAMask-HQ Dataset:")
list_folder_structure(dataset_path)

# Fetching 5000 images

In [ ]:
import os
import shutil

# 1. Define paths
# Adjust the base_path if your folder structure is slightly different
base_path = '/root/.cache/kagglehub/datasets/ipythonx/celebamaskhq/versions/1'
source_img_dir = os.path.join(base_path, 'CelebAMask-HQ/CelebA-HQ-img')
destination_dir = 'hair_data'
num_images_to_fetch = 5000

# 2. Setup destination
if not os.path.exists(destination_dir):
    os.makedirs(destination_dir)

# 3. Get list of files
# We sort them to ensure consistent results every time you run this
all_files = sorted([f for f in os.listdir(source_img_dir) if f.endswith(('.jpg', '.png'))])

# 4. Fetch the first 5,000
selected_files = all_files[:num_images_to_fetch]

print(f"Starting to copy {len(selected_files)} images to '{destination_dir}'...")

for filename in selected_files:
    src = os.path.join(source_img_dir, filename)
    dst = os.path.join(destination_dir, filename)

    # Use shutil.copy2 to preserve file metadata
    shutil.copy2(src, dst)

print(f"\nSuccess! {len(selected_files)} images copied to '{destination_dir}'.")

# Generating Labelled CSVs

In [ ]:
import pandas as pd
import os

# 1. SET THIS PATH: Point to the folder where your 5,000 images are actually located
source_img_dir = '/root/.cache/kagglehub/datasets/ipythonx/celebamaskhq/versions/1/CelebAMask-HQ/CelebA-HQ-img'

# 2. Get list of 5,000 files
all_files = sorted([f for f in os.listdir(source_img_dir) if f.endswith(('.jpg', '.png'))])
selected_files = all_files[:5000]

# 3. Create the DataFrame
# This map ensures your CSV points to the correct location in the Colab/local environment
data = []
for filename in selected_files:
    data.append({
        'image_name': filename,
        'image_path': os.path.join('/content/hair_data', filename)
    })

df = pd.DataFrame(data)

# 4. Save to CSV
df.to_csv('hair_data_metadata.csv', index=False)

print(f"Success! 'hair_data_metadata.csv' created with {len(df)} entries.")
print(df.head())

# Mapping Masked Images to Original Images

In [ ]:
import os
import shutil
import pandas as pd

# 1. Paths
# The CSV you created in the previous step
metadata_df = pd.read_csv('hair_data_metadata.csv')
source_mask_root = '/root/.cache/kagglehub/datasets/ipythonx/celebamaskhq/versions/1/CelebAMask-HQ/CelebAMask-HQ-mask-anno'
destination_mask_dir = 'hair_data/masks'

# 2. Setup
if not os.path.exists(destination_mask_dir):
    os.makedirs(destination_mask_dir)

# Extract just the image ID from the filename (e.g., '00001.jpg' -> '00001')
image_ids = [name.split('.')[0] for name in metadata_df['image_name']]

print(f"Searching for masks for {len(image_ids)} images...")

# 3. Recursive Search and Copy
mask_count = 0
for root, dirs, files in os.walk(source_mask_root):
    for filename in files:
        # Check if the mask file belongs to one of our target image IDs
        # We check if the start of the filename matches one of our IDs
        for img_id in image_ids:
            if filename.startswith(f"{img_id}_"):
                src = os.path.join(root, filename)
                dst = os.path.join(destination_mask_dir, filename)
                shutil.copy2(src, dst)
                mask_count += 1
                break # Move to next file once matched

print(f"\nSuccess! Found and copied {mask_count} mask files to '{destination_mask_dir}'.")

# Fetching Masked Images

In [ ]:
import os
import pandas as pd

# 1. Load your existing metadata
meta_df = pd.read_csv('hair_data_metadata.csv')

# 2. Paths
mask_dir = 'hair_data/masks'
output_csv = 'hair_mask_data.csv'

# 3. Collect mask data
mask_data = []

# List all masks we just downloaded
mask_files = [f for f in os.listdir(mask_dir) if f.endswith('.png')]

for mask_name in mask_files:
    # Example: mask_name is '00001_hair.png'
    # Split by '_' to get the image_id '00001'
    image_id = mask_name.split('_')[0]

    # Find the matching original image in our metadata
    # We look for the filename that starts with the image_id
    matching_image = meta_df[meta_df['image_name'].str.startswith(image_id)]

    if not matching_image.empty:
        orig_name = matching_image.iloc[0]['image_name']
        orig_path = matching_image.iloc[0]['image_path']

        mask_data.append({
            'mask_image_name': mask_name,
            'mask_image_path': os.path.join('/content/hair_data/masks', mask_name),
            'original_image_name': orig_name,
            'original_image_path': orig_path
        })

# 4. Save to CSV
mask_df = pd.DataFrame(mask_data)
mask_df.to_csv(output_csv, index=False)

print(f"Success! '{output_csv}' created with {len(mask_df)} correlations.")
print(mask_df.head())

In [ ]:
import pandas as pd

# 1. Load the existing CSV
df = pd.read_csv('hair_mask_data.csv')

# 2. Sort by 'original_image_name'
df_sorted = df.sort_values(by='original_image_name')

# 3. Save the sorted file (overwriting the original)
df_sorted.to_csv('hair_mask_data.csv', index=False)

print("Successfully sorted 'hair_mask_data.csv' by 'original_image_name'.")
print(df_sorted.head())

# Data Split

In [ ]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Paths
metadata_df = pd.read_csv('hair_mask_data.csv')
base_output_dir = 'master_hair_data'

# 2. Split Strategy: 70% Train, 15% Val, 15% Test
# We split based on unique original images
unique_images = metadata_df[['original_image_name', 'original_image_path']].drop_duplicates()
train_df, temp_df = train_test_split(unique_images, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# 3. Helper to organize and copy
def process_split(split_name, dataframe):
    split_dir = os.path.join(base_output_dir, split_name)
    img_dir = os.path.join(split_dir, 'images')
    mask_dir = os.path.join(split_dir, 'masks')
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(mask_dir, exist_ok=True)

    # Filter the main metadata for this split
    split_meta = metadata_df[metadata_df['original_image_name'].isin(dataframe['original_image_name'])]

    # Copy files
    for _, row in split_meta.iterrows():
        # Copy image
        shutil.copy2(row['original_image_path'], os.path.join(img_dir, row['original_image_name']))
        # Copy mask
        shutil.copy2(row['mask_image_path'], os.path.join(mask_dir, row['mask_image_name']))

    # Save the manifest for this split
    split_meta.to_csv(f'hair_{split_name}.csv', index=False)
    print(f"Processed {split_name}: {len(dataframe)} images and their masks.")

# Run the split
process_split('train', train_df)
process_split('val', val_df)
process_split('test', test_df)

print(f"\nSuccess! Dataset organized in '{base_output_dir}'.")
print("CSVs generated: hair_train.csv, hair_val.csv, hair_test.csv.")

# Clearing up memory

In [ ]:
import shutil
import os

# Define the list of folders you want to delete
folders_to_remove = [
    'master_hair_masks'
]

for folder in folders_to_remove:
    if os.path.exists(folder):
        # rmtree removes the directory and all its contents
        shutil.rmtree(folder)
        print(f"Successfully deleted: {folder}")
    else:
        print(f"Folder not found, skipping: {folder}")

# Fetching Data-files

In [ ]:
import os
import shutil

# 1. Define the source and destination
# If this path is wrong, use the print output below to find the correct one
source_dir = '/root/.cache/kagglehub/datasets/ipythonx/celebamaskhq/versions/1/CelebAMask-HQ'
destination_dir = 'contents'

# 2. Create destination
os.makedirs(destination_dir, exist_ok=True)

# 3. Process files
if os.path.exists(source_dir):
    txt_files = [f for f in os.listdir(source_dir) if f.endswith('.txt')]

    if txt_files:
        for txt_file in txt_files:
            shutil.copy2(os.path.join(source_dir, txt_file), os.path.join(destination_dir, txt_file))
            print(f"Copied: {txt_file}")
        print(f"Successfully moved {len(txt_files)} files to '{destination_dir}'.")
    else:
        print("No .txt files found in the source directory.")
else:
    print(f"Directory not found: {source_dir}")
    # HELPER: Print parent directory contents to help you locate the files
    parent = os.path.dirname(source_dir)
    if os.path.exists(parent):
        print(f"Available folders in parent path '{parent}': {os.listdir(parent)}")
    else:
        print("Parent directory not found either.")

In [ ]:
import pandas as pd
import io

# 1. Load the pose file
# Skipping the first 2 lines (header/metadata)
pose_file = 'contents/CelebAMask-HQ-pose-anno.txt'

# Read the file and create a DataFrame
# The file format is: filename  yaw  pitch  raw
pose_df = pd.read_csv(pose_file, sep='\s+', skiprows=2, names=['image_name', 'yaw', 'pitch', 'roll'])

# 2. Save for easy access
pose_df.to_csv('pose_data.csv', index=False)

print("Pose data successfully parsed into 'pose_data.csv'.")
print(pose_df.head())

# Listing Masks

In [ ]:
import pandas as pd
import os

# Your manifest files
csv_files = ['hair_train.csv', 'hair_val.csv', 'hair_test.csv']
all_unique_masks = set()

for csv in csv_files:
    if os.path.exists(csv):
        df = pd.read_csv(csv)
        # We iterate through the 'mask_image_name' column
        for name in df['mask_image_name']:
            # Example: '00001_hair.png'
            # 1. Split at the first underscore: ['00001', 'hair.png']
            # 2. Take the second part: 'hair.png'
            # 3. Remove the '.png' extension: 'hair'
            parts = name.split('_', 1)
            if len(parts) > 1:
                class_name = parts[1].replace('.png', '')
                all_unique_masks.add(class_name)
    else:
        print(f"Warning: {csv} not found, skipping...")

# Sort and print the result
sorted_masks = sorted(list(all_unique_masks))
print(f"Found {len(sorted_masks)} unique mask classes:")
for mask in sorted_masks:
    print(f"- {mask}")

# Selecting Required Masks

In [ ]:
# Mapping Dictionary
CLASS_MAP = {
    'hair': 1,
    'l_eye': 2,
    'r_eye': 3,
    'u_lip': 4,
    'l_lip': 5,
    'mouth': 6,
    'skin': 7,
    'cloth': 8
}

# Custom Model

# Dataset Loading, Model Definition and Training

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import pandas as pd
import os
import gc
import json
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# 1. Global Configuration
CLASS_MAP = {
    'hair': 1, 'l_eye': 2, 'r_eye': 3, 'u_lip': 4,
    'l_lip': 5, 'mouth': 6, 'skin': 7, 'cloth': 8
}
NUM_CLASSES = 9
# Device defined globally for consistent access across all functions
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Optimized Dataset
class FaceSegmentationDataset(Dataset):
    def __init__(self, csv_file, img_dir, mask_dir):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.mask_dir = mask_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = self.data.iloc[idx]['original_image_name']
        img_id = os.path.splitext(img_name)[0]

        img = cv2.resize(cv2.imread(os.path.join(self.img_dir, img_name)), (512, 512))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        img = torch.tensor(img.transpose(2, 0, 1), dtype=torch.float32)

        label_map = np.zeros((512, 512), dtype=np.int64)
        for class_name, class_id in CLASS_MAP.items():
            mask_path = os.path.join(self.mask_dir, f"{img_id}_{class_name}.png")
            if os.path.exists(mask_path):
                mask = cv2.resize(cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE), (512, 512))
                label_map[mask > 128] = class_id

        return img, torch.tensor(label_map, dtype=torch.long)

# 3. U-Net Architecture
class SimpleUNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.enc1 = nn.Conv2d(3, 32, 3, padding=1)
        self.enc2 = nn.Conv2d(32, 64, 3, padding=1)
        self.enc3 = nn.Conv2d(64, 128, 3, padding=1)
        self.dec1 = nn.Conv2d(128, 64, 3, padding=1)
        self.dec2 = nn.Conv2d(64, 32, 3, padding=1)
        self.final = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        x1 = F.relu(self.enc1(x))
        x2 = F.relu(self.enc2(F.max_pool2d(x1, 2)))
        x3 = F.relu(self.enc3(F.max_pool2d(x2, 2)))
        d1 = F.interpolate(x3, size=(256, 256), mode='bilinear', align_corners=False)
        d1 = F.relu(self.dec1(d1))
        d2 = F.interpolate(d1, size=(512, 512), mode='bilinear', align_corners=False)
        d2 = F.relu(self.dec2(d2))
        return self.final(d2)

# 4. Training Loop
def run_training(base_path, epochs=20):
    model = SimpleUNet(NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(FaceSegmentationDataset('hair_train.csv', f'{base_path}/train/images', f'{base_path}/train/masks'), batch_size=32, shuffle=True)
    val_loader = DataLoader(FaceSegmentationDataset('hair_val.csv', f'{base_path}/val/images', f'{base_path}/val/masks'), batch_size=32, shuffle=False)

    train_hist, val_hist = [], []

    for epoch in range(epochs):
        model.train()
        t_loss = 0.0
        for img, mask in train_loader:
            img, mask = img.to(DEVICE), mask.to(DEVICE)
            optimizer.zero_grad()
            out = model(img)
            loss = criterion(out, mask)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for img, mask in val_loader:
                img, mask = img.to(DEVICE), mask.to(DEVICE)
                v_loss += criterion(model(img), mask).item()

        avg_t, avg_v = t_loss/len(train_loader), v_loss/len(val_loader)
        train_hist.append(avg_t); val_hist.append(avg_v)
        print(f"Epoch {epoch+1} | Train: {avg_t:.4f} | Val: {avg_v:.4f}")

    with open('training_history.json', 'w') as f:
        json.dump({'train': train_hist, 'val': val_hist}, f)
    torch.save(model.state_dict(), 'segmentation_model.pth')
    print("Training finished. History saved.")


# Run Pipeline
run_training('/content/master_hair_data', epochs=20)

# Visualization

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np
import torch
import os

def visualize_performance(model, val_loader, device, history_file='training_history.json'):
    """
    Generates a side-by-side dashboard of Loss Curves and
    a Normalized Confusion Matrix for model evaluation.
    """
    # 1. Load History
    if not os.path.exists(history_file):
        print("Error: training_history.json not found. Please train the model first.")
        return
    with open(history_file, 'r') as f:
        history = json.load(f)

    # 2. Setup Plotting
    plt.figure(figsize=(16, 6))

    # Loss Plot
    plt.subplot(1, 2, 1)
    plt.plot(history['train'], label='Train Loss', color='blue', marker='o')
    plt.plot(history['val'], label='Val Loss', color='orange', linestyle='--', marker='x')
    plt.title('Training & Validation Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.legend(); plt.grid(True)

    # 3. Confusion Matrix Computation
    model.eval()
    all_preds, all_targets = [], []
    class_names = ['Bg', 'Hair', 'L_Eye', 'R_Eye', 'U_Lip', 'L_Lip', 'Mouth', 'Skin', 'Cloth']

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy().flatten()
            targets = masks.numpy().flatten()
            all_preds.extend(preds)
            all_targets.extend(targets)

    # Compute and Normalize Matrix
    cm = confusion_matrix(all_targets, all_preds, labels=range(9))
    cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-9)

    # Matrix Plot
    plt.subplot(1, 2, 2)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Normalized Confusion Matrix')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')

    plt.tight_layout()
    plt.show()

# Ensure model and loaders are initialized
model = SimpleUNet(NUM_CLASSES).to(DEVICE)
model.load_state_dict(torch.load('segmentation_model.pth'))

val_ds = FaceSegmentationDataset('hair_val.csv', '/content/sanity_check/val/images', '/content/sanity_check/val/masks')
val_loader = DataLoader(val_ds, batch_size=32)

# Generate dashboard
visualize_performance(model, val_loader, DEVICE)

# Testing & Visualization

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score
import os
import json

def generate_full_performance_report(model, test_img_dir, test_mask_dir, history_file='training_history.json'):
    # 1. Load History
    with open(history_file, 'r') as f:
        history = json.load(f)

    model.eval()
    all_preds, all_targets = [], []

    # 2. Collect Predictions for Confusion Matrix
    img_files = [f for f in os.listdir(test_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    with torch.no_grad():
        for img_name in img_files:
            # Load and preprocess
            img_path = os.path.join(test_img_dir, img_name)
            img = cv2.resize(cv2.imread(img_path), (512, 512))
            img_t = torch.tensor(cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

            # Predict
            out = model(img_t)
            pred = torch.argmax(out, dim=1).cpu().numpy().flatten()

            # Target
            mask_path = os.path.join(test_mask_dir, os.path.splitext(img_name)[0] + ".png")
            if os.path.exists(mask_path):
                target = cv2.resize(cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE), (512, 512)).flatten()
                all_preds.extend(pred)
                all_targets.extend(target)

    # 3. Visualize
    fig = plt.figure(figsize=(18, 5))

    # Plot Loss
    plt.subplot(1, 3, 1)
    plt.plot(history['train'], label='Train Loss'); plt.plot(history['val'], label='Val Loss')
    plt.title('Training History'); plt.legend(); plt.grid(True)

    # Plot Confusion Matrix
    cm = confusion_matrix(all_targets, all_preds, labels=range(9))
    cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-9)
    plt.subplot(1, 3, 2)
    sns.heatmap(cm_norm, annot=False, cmap='Blues', xticklabels=range(9), yticklabels=range(9))
    plt.title('Normalized Confusion Matrix')

    # Plot Accuracy Bar
    acc = accuracy_score(all_targets, all_preds)
    plt.subplot(1, 3, 3)
    plt.bar(['Test Accuracy'], [acc], color='green')
    plt.ylim(0, 1); plt.title(f'Test Accuracy: {acc:.2%}')

    plt.tight_layout()
    plt.show()

# --- CALLING THE FUNCTION ---
generate_full_performance_report(model, '/content/master_hair_data/test/images', '/content/master_hair_data/test/masks')

# Downloading Dataset

In [ ]:
import shutil
from IPython.display import FileLink

# Define the folder to zip
folder_to_zip = '/content/master_hair_data'
output_filename = 'master_hair_data.zip'

# Create zip file
shutil.make_archive(output_filename.replace('.zip', ''), 'zip', folder_to_zip)

# Generate download link
display(FileLink(output_filename))

# **Gender Model**

# Downloading data

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jangedoo/utkface-new")

print("Path to dataset files:", path)

# Fetching 5000 images

In [ ]:
import os
import shutil
from tqdm import tqdm

def organize_utkface(source_dir, dest_dir, limit=2500):
    # Setup subfolders
    male_dir = os.path.join(dest_dir, 'male')
    female_dir = os.path.join(dest_dir, 'female')
    os.makedirs(male_dir, exist_ok=True)
    os.makedirs(female_dir, exist_ok=True)

    male_count = 0
    female_count = 0

    # Iterate through files
    files = [f for f in os.listdir(source_dir) if f.endswith('.jpg')]

    print("Organizing images...")
    for filename in tqdm(files):
        # Format: age_gender_race_date.jpg
        try:
            parts = filename.split('_')
            if len(parts) < 2: continue

            gender = parts[1] # 0 for male, 1 for female

            if gender == '0' and male_count < limit:
                shutil.copy(os.path.join(source_dir, filename), os.path.join(male_dir, filename))
                male_count += 1
            elif gender == '1' and female_count < limit:
                shutil.copy(os.path.join(source_dir, filename), os.path.join(female_dir, filename))
                female_count += 1

        except Exception as e:
            print(f"Skipping {filename}: {e}")

        if male_count >= limit and female_count >= limit:
            break

    print(f"Finished! Copied {male_count} male and {female_count} female images.")

# --- Execution ---
source_path = '/kaggle/input/utkface-new/UTKFace'
destination_path = './gender_data'
organize_utkface(source_path, destination_path, limit=2500)

# Generating CSV

In [ ]:
import os
import pandas as pd

def create_gender_csv(root_dir, output_csv='gender_metadata.csv'):
    data = []

    # Iterate through 'male' and 'female' subfolders
    for label in ['male', 'female']:
        label_dir = os.path.join(root_dir, label)
        # Convert folder name to label: 0 for male, 1 for female
        gender_label = 0 if label == 'male' else 1

        if os.path.exists(label_dir):
            for filename in os.listdir(label_dir):
                if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    # Create dictionary of image info
                    data.append({
                        'image_name': filename,
                        'path': os.path.join(label_dir, filename),
                        'gender_label': gender_label
                    })

    # Create DataFrame and save
    df = pd.DataFrame(data)
    df.to_csv(output_csv, index=False)
    print(f"Successfully created {output_csv} with {len(df)} entries.")
    return df

# --- Execution ---
# Assumes your folder is named 'gender_data'
df = create_gender_csv('gender_data')

# Display the first few rows
print(df.head())

# Dataset Split

In [ ]:
import os
import shutil
import random
from math import floor

def split_dataset(source_dir, dest_dir, train_ratio=0.8, val_ratio=0.1):
    # Create structure
    for split in ['train', 'val', 'test']:
        for gender in ['male', 'female']:
            os.makedirs(os.path.join(dest_dir, split, gender), exist_ok=True)

    for gender in ['male', 'female']:
        gender_path = os.path.join(source_dir, gender)
        files = [f for f in os.listdir(gender_path) if f.lower().endswith('.jpg')]
        random.shuffle(files)

        # Calculate split indices
        n = len(files)
        train_idx = floor(n * train_ratio)
        val_idx = floor(n * (train_ratio + val_ratio))

        # Distribute files
        for i, f in enumerate(files):
            if i < train_idx:
                split = 'train'
            elif i < val_idx:
                split = 'val'
            else:
                split = 'test'

            shutil.copy(os.path.join(gender_path, f), os.path.join(dest_dir, split, gender, f))

    print("Dataset split successfully into train, val, and test folders.")

# --- Execution ---
# Assumes your source is the 'gender_data' folder created earlier
split_dataset('gender_data', 'split_gender_data')

In [ ]:
import os
import pandas as pd

def create_split_csvs(root_dir, output_dir='/content/'):
    os.makedirs(output_dir, exist_ok=True)

    # Iterate over splits
    for split in ['train', 'val', 'test']:
        split_path = os.path.join(root_dir, split)
        data = []

        # Iterate over genders
        for gender in ['male', 'female']:
            gender_dir = os.path.join(split_path, gender)
            label = 0 if gender == 'male' else 1

            if os.path.exists(gender_dir):
                for filename in os.listdir(gender_dir):
                    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                        data.append({
                            'image_name': filename,
                            'path': os.path.join(gender_dir, filename),
                            'gender_label': label
                        })

        # Save split CSV
        df = pd.DataFrame(data)
        csv_name = os.path.join(output_dir, f'{split}_gender.csv')
        df.to_csv(csv_name, index=False)
        print(f"Created {csv_name} with {len(df)} images.")

# --- Execution ---
# Assumes your split data is in 'split_gender_data'
create_split_csvs('split_gender_data')

# Custom Model

# Data Loading

In [ ]:
import os
import cv2
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

class GenderDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        """
        Args:
            csv_file (string): Path to the csv file.
            transform (callable, optional): Optional transform to be applied
                on a sample.
        """
        self.df = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. Access path and label
        path = self.df.iloc[idx]['path']
        label = self.df.iloc[idx]['gender_label']

        # 2. Robust Image Loading
        # Check if file exists to prevent hard crashes during training
        if not os.path.exists(path):
            raise FileNotFoundError(f"Image not found at: {path}")

        # Load with OpenCV (BGR)
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            raise ValueError(f"Could not read image at: {path}. Check file integrity.")

        # Convert BGR (OpenCV default) to RGB
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # 3. CONVERSION: Ensure it is a PIL Image
        # PIL.Image.fromarray handles numpy arrays perfectly
        img_pil = Image.fromarray(img_rgb)

        # 4. Apply transforms
        if self.transform:
            img_pil = self.transform(img_pil)

        return img_pil, label

# Building Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class GenderCNN(nn.Module):
    def __init__(self):
        super(GenderCNN, self).__init__()

        # Convolutional Block 1: Extracts basic features (edges, textures)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # Convolutional Block 2: Extracts complex features
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Convolutional Block 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        # Fully Connected Layers
        # After 3 pooling operations, the 128x128 image becomes 16x16
        self.fc1 = nn.Linear(128 * 16 * 16, 512)
        self.fc2 = nn.Linear(512, 2) # 2 classes: Male and Female

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        # Apply conv layers with ReLU and Pooling
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        # Flatten the tensor for the fully connected layers
        x = x.view(-1, 128 * 16 * 16)

        # Apply FC layers with dropout to prevent overfitting
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

# Model Training

In [ ]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms

# Ensure DEVICE is defined globally or passed as an argument
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Ensure GenderCNN is defined in your environment
model = GenderCNN().to(DEVICE)

def train_gender_model(train_csv, val_csv, epochs=20, save_path='gender_model.pth'):
    # 1. Initialization: Removed ToPILImage() as dataset returns PIL objects
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])


    train_loader = DataLoader(GenderDataset(train_csv, transform=transform), batch_size=32, shuffle=True)
    val_loader = DataLoader(GenderDataset(val_csv, transform=transform), batch_size=32)


    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')

    # 2. Training Loop
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation Phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                val_loss += criterion(outputs, labels).item()

        # Calculate Averages
        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)

        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)

        print(f"Epoch {epoch+1} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

        # 3. Save Best Model
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), save_path)
            print(f"-> Model saved at epoch {epoch+1}")

    # 4. Save History
    with open('training_history.json', 'w') as f:
        json.dump(history, f)
    print("Training Complete. History and Model saved.")

# --- Start Training ---
# Ensure your CSV paths are correct as per your file system
train_gender_model('/content/train_gender.csv', '/content/val_gender.csv')

# Visualization

In [ ]:
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader
from torchvision import transforms

def visualize_val_performance(model, val_csv, history_file='training_history.json'):
    with open(history_file, 'r') as f:
        history = json.load(f)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Plot Loss
    train_vals = history.get('train', history.get('train_loss', []))
    val_vals = history.get('val', history.get('val_loss', []))
    axes[0].plot(train_vals, label='Train Loss'); axes[0].plot(val_vals, label='Val Loss')
    axes[0].legend(); axes[0].grid(True)

    # 3. Compute Confusion Matrix
    model.eval()
    all_preds, all_labels = [], []

    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    val_ds = GenderDataset(val_csv, transform=transform)
    val_loader = DataLoader(val_ds, batch_size=32)

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)

            # --- CRITICAL FIX ---
            # If outputs are [Batch, Channels, Height, Width],
            # we MUST reduce it to [Batch, Classes] before argmax.
            if outputs.dim() == 4:
                outputs = torch.mean(outputs, dim=[2, 3]) # Global Average Pooling

            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy().flatten())
            all_labels.extend(labels.numpy().flatten())

    # Ensure integer types for sklearn
    y_true = np.array(all_labels, dtype=int)
    y_pred = np.array(all_preds, dtype=int)

    cm = confusion_matrix(y_true, y_pred)

    # 4. Plot Confusion Matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                xticklabels=['Male', 'Female'], yticklabels=['Male', 'Female'])
    axes[1].set_title('Confusion Matrix (Validation Set)')
    plt.tight_layout()
    plt.show()

# --- Execution ---

# Load the trained weights from your saved file
# map_location ensures it works even if you trained on GPU but are running on CPU
model.load_state_dict(torch.load('gender_model.pth', map_location=DEVICE))

# Now you can safely call the visualization
visualize_val_performance(model, '/content/val_gender.csv')

# Testing & Visualization

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score
from torch.utils.data import DataLoader
from torchvision import transforms

def evaluate_on_test_set(model, test_csv):
    model.eval()

    # 1. Setup test data loader (Removed ToPILImage)
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    test_ds = GenderDataset(test_csv, transform=transform)
    test_loader = DataLoader(test_ds, batch_size=32)

    all_preds, all_labels = [], []
    test_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    # 2. Evaluation Loop
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)

            # CRITICAL FIX: Ensure output is [Batch, Classes] if it is [Batch, Classes, H, W]
            if outputs.dim() == 4:
                outputs = outputs.view(outputs.size(0), outputs.size(1), -1).mean(dim=2)

            loss = criterion(outputs, labels)
            test_loss += loss.item()

            # Flatten predictions and labels for sklearn
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy().flatten().tolist())
            all_labels.extend(labels.cpu().numpy().flatten().tolist())

    # 3. Calculate Metrics with explicit integer types
    avg_loss = test_loss / len(test_loader)
    y_true = np.array(all_labels, dtype=int)
    y_pred = np.array(all_preds, dtype=int)

    accuracy = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    # 4. Visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    axes[0].bar(['Loss', 'Accuracy'], [avg_loss, accuracy], color=['red', 'green'])
    axes[0].set_title(f'Test Performance | Acc: {accuracy:.2%}')
    axes[0].set_ylim(0, 1.2)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[1],
                xticklabels=['Male', 'Female'], yticklabels=['Male', 'Female'])
    axes[1].set_title('Confusion Matrix (Test Data)')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

    plt.tight_layout()
    plt.show()

# --- Execution ---
# Ensure your model is loaded
# Ensure DEVICE, GenderDataset, and GenderCNN are defined
model = GenderCNN().to(DEVICE)
model.load_state_dict(torch.load('gender_model.pth', map_location=DEVICE))

# Run evaluation
evaluate_on_test_set(model, '/content/test_gender.csv')

# Downloading Dataset

In [ ]:
import shutil
from IPython.display import FileLink

# Define the folder to zip
folder_to_zip = '/content/split_gender_data'
output_filename = 'split_gender_data'

# Create zip file
shutil.make_archive(output_filename.replace('.zip', ''), 'zip', folder_to_zip)



# **Github Repo**

https://github.com/KVAlwaysLearning/Long_Hair_Detection_Sub

# **Streamlit App**

https://longhairdetectionsub-app.streamlit.app/